# Deployment: the pipeline is the artifact

Here the deployment unit isn't a model or an endpoint — it's the
**pipeline definition**. A `LocalPipelineSession` runs the same steps
SageMaker Pipelines would: train → evaluate → condition → register.

In [ ]:
import boto3
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.local.local_session import LocalPipelineSession
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import TrainingStep

BUCKET = "wine-quality-classification"
pipeline_session = LocalPipelineSession()
estimator = Estimator(
    image_uri="wine-quality-classification:train",
    role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
    instance_count=1,
    instance_type="local",
    output_path=f"s3://{BUCKET}/output",
    sagemaker_session=pipeline_session,
    hyperparameters={"C": 1.0, "max_iter": 2000},
)
estimator.set_entry_point(
    source_dir="src/wine_quality_classification", entry_point="train.py"
)
step_train = TrainingStep(
    name="TrainCultivarClassifier",
    estimator=estimator,
    inputs={"train": TrainingInput(s3_data=f"s3://{BUCKET}/input/train")},
)
pipeline = Pipeline(
    name="wine-cultivar-pipeline",
    steps=[step_train],
    sagemaker_session=pipeline_session,
)
pipeline.upsert(
    role_arn="arn:aws:iam::111111111111:role/service-role/SageMakerRole"
)

## Run it

In [ ]:
execution = pipeline.start()
execution.arn

In [ ]:
execution.describe()["PipelineExecutionStatus"]

## Inspect the step's output

Every step is a normal training job — artifacts land under `output/` on
moto, so the run is fully inspectable locally.

In [ ]:
s3 = boto3.client(
    "s3", endpoint_url="http://wine-quality-classification-moto:5000"
)
sorted(
    o["Key"]
    for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="output/").get(
        "Contents", []
    )
)[-4:]

## Where Feast fits

The channel this pipeline consumes came from `get_historical_features` —
swap the static `TrainingInput` for a retrieval step and the pipeline
becomes fully reproducible against the store. That swap is the production
pattern; here it stays explicit so the mechanics are visible.